# EEG Feature Extraction and Visualization - Tutorial

This notebook demonstrates how to use the comprehensive feature extraction and visualization tools.

## Features Extracted:
- **Temporal**: Statistical measures, Hjorth parameters, zero-crossing rate, RMS
- **Spectral**: Band powers (delta, theta, alpha, beta, gamma), spectral entropy, dominant frequency
- **Functional**: Correlation-based connectivity, Phase Locking Value (PLV)

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: /Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech


## Step 1: Extract Features

Run the feature extraction script (this may take a few minutes):

In [3]:
# Import feature extraction functions
from scripts.features.comprehensive_features import (
    iterate_all_files_and_extract,
    extract_temporal_features,
    extract_spectral_features,
    extract_functional_features
)

# Set paths
meta_csv = project_root / "data" / "interim" / "eeg_metadata.csv"
eloc_path = project_root / "scripts" / "data_processing" / "Preprocessing" / "ebneuro.eloc"
output_csv = project_root / "data" / "interim" / "comprehensive_features.csv"

# Check if features already exist
if output_csv.exists():
    print(f"Features already extracted: {output_csv}")
    df_features = pd.read_csv(output_csv)
else:
    print("Extracting features... (this may take several minutes)")
    df_features = iterate_all_files_and_extract(meta_csv, eloc_path=eloc_path)
    df_features.to_csv(output_csv, index=False)
    print(f"✓ Features saved to {output_csv}")

print(f"\nDataFrame shape: {df_features.shape}")
print(f"Columns: {df_features.columns.tolist()[:10]}...")

Extracting features... (this may take several minutes)

Processing 00 | 01: 00_01.h5
Not setting metadata
110 matching events found
No baseline correction applied
0 projection items activated
Adding metadata with 3 columns
  Processed 50/110 epochs...
  Processed 100/110 epochs...
  Extracted 6710 feature rows

Processing 00 | 02: 00_02.h5
Not setting metadata
110 matching events found
No baseline correction applied
0 projection items activated
Adding metadata with 3 columns
  Processed 50/110 epochs...
  Processed 100/110 epochs...
  Extracted 6710 feature rows

Processing 00 | 03: 00_03.h5
Not setting metadata
110 matching events found
No baseline correction applied
0 projection items activated
Adding metadata with 3 columns
  Processed 50/110 epochs...
  Processed 100/110 epochs...
  Extracted 6710 feature rows

Processing 00 | 04: 00_04.h5
Not setting metadata
110 matching events found
No baseline correction applied
0 projection items activated
Adding metadata with 3 columns
  Proc

ValueError: metadata must have the same number of rows (113) as events (110)

## Step 2: Explore the Features DataFrame

In [ ]:
# Display first few rows
df_features.head()

In [ ]:
# Display feature categories
temporal_features = [col for col in df_features.columns if col.startswith('temp_')]
spectral_features = [col for col in df_features.columns if col.startswith('spec_')]
functional_features = [col for col in df_features.columns if col.startswith('func_')]

print(f"Temporal features ({len(temporal_features)}): {temporal_features}")
print(f"\nSpectral features ({len(spectral_features)}): {spectral_features}")
print(f"\nFunctional features ({len(functional_features)}): {functional_features}")

In [ ]:
# Summary statistics
print("Dataset Summary:")
print(f"Number of subjects: {df_features['subject_id'].nunique()}")
print(f"Number of sessions: {df_features['session_id'].nunique()}")
print(f"Number of epochs: {df_features['epoch_idx'].nunique()}")
print(f"Number of channels: {df_features['channel'].nunique()}")
print(f"Number of labels: {df_features['label_name'].nunique()}")
print(f"\nTotal feature vectors: {len(df_features)}")

## Step 3: Basic Feature Analysis

In [ ]:
# Analyze power distribution across channels
channel_avg_power = df_features.groupby('channel')['spec_total_power'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
plt.bar(range(len(channel_avg_power)), channel_avg_power.values, color='steelblue', alpha=0.7)
plt.xticks(range(len(channel_avg_power)), channel_avg_power.index, rotation=90, fontsize=8)
plt.xlabel('Channel', fontsize=12)
plt.ylabel('Average Total Power (µV²)', fontsize=12)
plt.title('Average Power Across Channels', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nTop 10 highest power channels:")
print(channel_avg_power.head(10))

In [ ]:
# Compare band powers
band_cols = ['spec_delta_rel', 'spec_theta_rel', 'spec_alpha_rel', 'spec_beta_rel', 'spec_gamma_rel']
band_averages = df_features[band_cols].mean()

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#9b59b6', '#2ecc71', '#e74c3c', '#f39c12']
plt.bar(['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma'], band_averages.values, color=colors, alpha=0.7)
plt.ylabel('Average Relative Power', fontsize=12)
plt.title('Average Relative Band Power Distribution', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 4: Generate Visualizations

In [ ]:
# Import visualization functions
from scripts.graphs.feature_visualizations import (
    plot_electrode_power_across_epochs,
    plot_top_electrodes_per_epoch,
    plot_feature_distributions,
    plot_feature_correlation_matrix
)

# Create output directory
output_dir = project_root / "figures" / "feature_visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving visualizations to: {output_dir}")

In [ ]:
# Visualization 1: Single electrode power variation
subject_id = df_features['subject_id'].iloc[0]
electrode = 'Cz'  # Central electrode

if electrode in df_features['channel'].values:
    plot_electrode_power_across_epochs(
        df_features, 
        electrode=electrode, 
        subject_id=subject_id,
        output_path=output_dir / f"power_variation_{electrode}.png"
    )
    print(f"✓ Created power variation plot for {electrode}")

In [ ]:
# Visualization 2: Top electrodes per epoch
plot_top_electrodes_per_epoch(
    df_features, 
    subject_id=subject_id, 
    top_n=5,
    output_path=output_dir / "top_electrodes_per_epoch.png"
)
print("✓ Created top electrodes heatmap")

In [ ]:
# Visualization 3: Feature distributions
plot_feature_distributions(
    df_features, 
    output_path=output_dir / "feature_distributions.png"
)
print("✓ Created feature distribution plots")

In [ ]:
# Visualization 4: Feature correlation matrix
plot_feature_correlation_matrix(
    df_features, 
    output_path=output_dir / "feature_correlation_matrix.png"
)
print("✓ Created feature correlation matrix")

## Step 5: Advanced Analysis - Label Comparison

In [ ]:
# Compare features across different labels
labels_to_compare = df_features['label_name'].value_counts().head(5).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

features_to_plot = [
    ('spec_alpha_rel', 'Alpha Relative Power'),
    ('spec_beta_rel', 'Beta Relative Power'),
    ('temp_hjorth_mobility', 'Hjorth Mobility'),
    ('func_mean_corr', 'Mean Correlation')
]

for i, (feat, title) in enumerate(features_to_plot):
    if feat in df_features.columns:
        data_to_plot = [df_features[df_features['label_name'] == label][feat].dropna().values 
                       for label in labels_to_compare]
        
        axes[i].boxplot(data_to_plot, labels=labels_to_compare)
        axes[i].set_ylabel(title, fontsize=10)
        axes[i].set_xticklabels(labels_to_compare, rotation=45, ha='right', fontsize=9)
        axes[i].set_title(f'{title} by Label', fontsize=11, fontweight='bold')
        axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Feature Comparison Across Top 5 Labels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / "label_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Created label comparison plot")

## Step 6: Export Summary Report

In [ ]:
# Create a summary report
summary = {
    'Dataset Info': {
        'Total Subjects': df_features['subject_id'].nunique(),
        'Total Sessions': df_features['session_id'].nunique(),
        'Total Epochs': df_features['epoch_idx'].nunique(),
        'Total Channels': df_features['channel'].nunique(),
        'Total Labels': df_features['label_name'].nunique(),
        'Total Feature Vectors': len(df_features)
    },
    'Feature Counts': {
        'Temporal Features': len(temporal_features),
        'Spectral Features': len(spectral_features),
        'Functional Features': len(functional_features),
        'Total Features': len(temporal_features) + len(spectral_features) + len(functional_features)
    },
    'Average Power Statistics': {
        'Mean Total Power': df_features['spec_total_power'].mean(),
        'Std Total Power': df_features['spec_total_power'].std(),
        'Max Total Power': df_features['spec_total_power'].max(),
        'Min Total Power': df_features['spec_total_power'].min()
    },
    'Band Power Averages': {
        'Delta': df_features['spec_delta_rel'].mean(),
        'Theta': df_features['spec_theta_rel'].mean(),
        'Alpha': df_features['spec_alpha_rel'].mean(),
        'Beta': df_features['spec_beta_rel'].mean(),
        'Gamma': df_features['spec_gamma_rel'].mean()
    }
}

# Print summary
import json
print(json.dumps(summary, indent=2))

# Save to file
with open(output_dir / 'feature_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Summary saved to {output_dir / 'feature_summary.json'}")

## Conclusion

This notebook demonstrated:
1. ✅ Feature extraction from EEG data (temporal, spectral, functional)
2. ✅ Creating a unified features dataframe
3. ✅ Visualizing power variations across electrodes and epochs
4. ✅ Identifying high-power electrodes
5. ✅ Comparing features across different labels
6. ✅ Analyzing feature distributions and correlations

All visualizations are saved in: `figures/feature_visualizations/`